# 02 ResNet 分类与临床融合
读取四层缓存，训练当前的 ResNet50 分类网络，再训练临床对照和临床融合模型。结果保存到 `results/resnet`，06 直接从这里读取最终权重。

这本只保留当前模型。已有旧实验可以保存它的测试预测 CSV，后续放到 06 中比较。


## 这次调整了哪些地方
- 将 ResNet50 的每层影像特征从 2048 维压缩到 256 维，减少后续分类参数。
- 病灶分支使用小型卷积编码器和 GroupNorm，输出 64 维特征。
- 四层之间采用门控注意力，学习不同层的权重；加入 LayerNorm 和 Dropout。
- 前 3 轮先训练新增部分，之后只微调 ResNet 的最后一组残差块；固定预训练 BatchNorm，主干使用更小的学习率。
- 训练轮数在每个外层训练集内部选择，再重新训练该折模型。外层验证患者只用于生成折外预测。

仍然使用三序列、四层、224×224 输入，以及原来的几何和强度增强。调整效果需要看实验结果，这里不预设 AUC 会提高。


## 路径、数据与训练参数


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "config.json").exists() and (ROOT / "1117" / "config.json").exists():
    ROOT = ROOT / "1117"
cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))

def path_from_config(key):
    value = Path(cfg[key]).expanduser()
    return value if value.is_absolute() else ROOT / value

CACHE = path_from_config("cache_path")
COHORT = path_from_config("cohort_path")
RESULTS = path_from_config("results_dir")
SEED = 20260922
CLINICAL = ["age", "education", "nihss", "hcy", "ua"]
SEQUENCES = ["t1", "t2", "dwi"]
IMAGE_FILES = ["t1_flair.nii.gz", "t2.nii.gz", "dwi_b1000.nii.gz"]
print("患者清单：", COHORT)
print("结果目录：", RESULTS)


In [ ]:
def read_cases():
    cases = pd.read_csv(COHORT, dtype={"mr": str})
    if not cases.mr.is_unique or not set(cases.split) <= {"development", "validation"}:
        raise ValueError("患者编号重复或分组有误，请检查cohort.csv。")
    if not set(cases.label) <= {0, 1} or cases.mr.isna().any():
        raise ValueError("患者编号缺失或结局不是0/1。")
    return cases


def read_classification_data():
    cases = read_cases()
    with np.load(CACHE, allow_pickle=False) as data:
        cache_mrs = data["mrs"].astype(str).tolist()
        if len(set(cache_mrs)) != len(cache_mrs) or set(cache_mrs) != set(cases.mr):
            raise ValueError("缓存与患者清单的MR编号不一致。")
        lookup = {mr: i for i, mr in enumerate(cache_mrs)}
        order = [lookup[mr] for mr in cases.mr]
        if not np.array_equal(data["labels"][order], cases.label) or not np.array_equal(data["max_layers"][order], cases.max_layer):
            raise ValueError("患者标签或取层与缓存不一致。")
        images = data["images"][order].astype(np.float32)
        masks = data["masks"][order].astype(np.uint8)
    if images.shape[1:] != (4, 3, 224, 224) or masks.shape != images.shape:
        raise ValueError("分类输入应为每例4层、3通道、224×224。")
    if not np.isfinite(images).all() or not np.isin(masks, [0, 1]).all():
        raise ValueError("影像存在无效值或mask不是0/1。")
    return cases, images, masks


In [ ]:
cases, images, masks = read_classification_data()
OUTPUT = RESULTS / "resnet"
TRAIN = {
    "batch_size": 16,
    "max_epochs": 30,
    "patience": 8,
    "warmup_epochs": 3,
    "backbone_lr": 1e-5,
    "head_lr": 5e-4,
}
MODEL = {"image_dim": 256, "mask_dim": 64, "attention_dim": 128,
         "use_mask": True, "use_attention": True}
RUN_TRAINING = False
RUN_CLINICAL = False
RESTART = False

display(cases.groupby("split").agg(n=("mr", "size"), positive=("label", "sum")))


## 网络结构与预测
同一套 ResNet50 逐层编码三序列影像，病灶编码器处理三个标签通道，拼接后对四层加权汇总，输出一个患者级概率。

训练时载入 ImageNet 初始权重；已有缓存可直接使用，否则 torchvision 会下载一次。测试只读取本实验保存的权重。


In [ ]:
import random
import torch
from torch import nn
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import affine
from torchvision.transforms import InterpolationMode

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("计算设备：", DEVICE)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class GatedAttention(nn.Module):
    def __init__(self, features, hidden=128):
        super().__init__()
        self.value = nn.Linear(features, hidden)
        self.gate = nn.Linear(features, hidden)
        self.score = nn.Linear(hidden, 1)

    def forward(self, x):
        return self.score(torch.tanh(self.value(x)) * torch.sigmoid(self.gate(x)))


class MaskEncoder(nn.Module):
    def __init__(self, features=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, 2, 1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1, bias=False), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1, bias=False), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, features),
            nn.LayerNorm(features), nn.GELU(), nn.Dropout(.3))

    def forward(self, x):
        return self.net(x.float())


class ResNetClassifier(nn.Module):
    def __init__(self, image_dim=256, mask_dim=64, attention_dim=128,
                 use_mask=True, use_attention=True, pretrained=False):
        super().__init__()
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.image_features = nn.Sequential(*list(backbone.children())[:-1])
        self.image_projection = nn.Sequential(nn.Linear(2048, image_dim), nn.LayerNorm(image_dim), nn.GELU(), nn.Dropout(.3))
        self.mask_features = MaskEncoder(mask_dim) if use_mask else None
        width = image_dim + (mask_dim if use_mask else 0)
        self.attention = GatedAttention(width, attention_dim) if use_attention else None
        self.head = nn.Sequential(nn.LayerNorm(width), nn.Dropout(.4), nn.Linear(width, 1))

    def train(self, mode=True):
        super().train(mode)
        for layer in self.image_features.modules():
            if isinstance(layer, nn.modules.batchnorm._BatchNorm):
                layer.eval()
        return self

    def set_stage(self, fine_tune):
        for parameter in self.image_features.parameters():
            parameter.requires_grad_(False)
        if fine_tune:
            for parameter in self.image_features[7].parameters():
                parameter.requires_grad_(True)
        for layer in self.image_features.modules():
            if isinstance(layer, nn.modules.batchnorm._BatchNorm):
                for parameter in layer.parameters():
                    parameter.requires_grad_(False)

    def forward(self, images, masks):
        b, slices, channels, h, w = images.shape
        features = self.image_projection(self.image_features(images.reshape(b * slices, channels, h, w)).flatten(1))
        features = features.reshape(b, slices, -1)
        if self.mask_features is not None:
            lesion = self.mask_features(masks.reshape(b * slices, 3, h, w)).reshape(b, slices, -1)
            features = torch.cat([features, lesion], dim=-1)
        if self.attention is None:
            pooled = features.mean(dim=1)
        else:
            weights = self.attention(features).softmax(dim=1)
            pooled = (features * weights).sum(dim=1)
        return self.head(pooled).squeeze(-1)


def sequence_input(images, sequence="all"):
    if sequence == "all":
        return images
    channel = SEQUENCES.index(sequence)
    return images[:, :, [channel, channel, channel]]


@torch.no_grad()
def predict(model, images, masks, batch_size=16):
    model.eval()
    probabilities = []
    for start in range(0, len(images), batch_size):
        x = torch.as_tensor(images[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        m = torch.as_tensor(masks[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(x, m)
        probabilities.append(logits.float().sigmoid().cpu().numpy())
    return np.concatenate(probabilities)


def load_classifier(file):
    saved = torch.load(file, map_location="cpu", weights_only=True)
    model = ResNetClassifier(**saved["model_args"], pretrained=False).to(DEVICE)
    model.load_state_dict(saved["state_dict"])
    return model.eval(), saved


## 数据增强与评价指标
影像和标签采用相同的旋转、平移和缩放，四层保持一致；标签使用最近邻插值。亮度和噪声仅加到影像上。分类阈值从开发队列的折外预测选择，测试阶段沿用。


In [ ]:
def augment_batch(images: torch.Tensor, masks: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    out_images = images.clone()
    out_masks = masks.clone()
    B, S = images.shape[:2]
    for b in range(B):
        angle = random.uniform(-6.0, 6.0)
        translate = (random.uniform(-6.0, 6.0), random.uniform(-6.0, 6.0))
        scale = random.uniform(0.96, 1.04)
        for s in range(S):
            out_images[b, s] = affine(
                images[b, s],
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.BILINEAR,
                fill=0.0,
            )
            out_masks[b, s] = affine(
                masks[b, s].float(),
                angle=angle,
                translate=translate,
                scale=scale,
                shear=0.0,
                interpolation=InterpolationMode.NEAREST,
                fill=0.0,
            ).byte()

    jitter = 0.9 + 0.2 * torch.rand(B, 1, 1, 1, 1, device=out_images.device)
    out_images = out_images * jitter
    if random.random() < 0.5:
        out_images = out_images + torch.randn_like(out_images) * 0.008
    out_images = out_images.clamp(0.0, 1.0)
    return out_images, out_masks


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss


def choose_threshold(labels, probabilities):
    fpr, tpr, thresholds = roc_curve(labels, probabilities)
    finite = np.isfinite(thresholds)
    return float(thresholds[finite][np.argmax((tpr - fpr)[finite])])


def classification_metrics(labels, probabilities, threshold):
    labels, probabilities = np.asarray(labels, int), np.asarray(probabilities, float)
    if not np.isfinite(threshold):
        raise ValueError("分类阈值必须是开发队列确定的有限数值。")
    if set(labels) != {0, 1} or not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):
        raise ValueError("指标需要两类结局，以及0～1之间的有效概率。")
    tn, fp, fn, tp = confusion_matrix(labels, probabilities >= threshold, labels=[0, 1]).ravel()
    return {"n": len(labels), "auc": float(roc_auc_score(labels, probabilities)), "threshold": float(threshold),
            "accuracy": float((tp + tn) / len(labels)), "sensitivity": float(tp / (tp + fn)),
            "specificity": float(tn / (tn + fp)), "brier": float(brier_score_loss(labels, probabilities))}


## 五折训练和最终模型
每折先在训练患者中划出 20% 选择轮数，再用该折全部训练患者重训。最后取五折最佳轮数的中位数，使用全部开发患者训练 `model.pt`。

已完成的折会读取现有结果；中断在某一折时，该折会重新训练。改过参数或模型后，将 `RESTART=True` 重训；想保留上次实验，就修改 `OUTPUT` 为另一个文件夹。


In [ ]:
from sklearn.model_selection import train_test_split


def fit_network(images, masks, labels, train_index, model_args, settings, epochs, seed, valid_index=None):
    set_seed(seed)
    model = ResNetClassifier(**model_args, pretrained=True).to(DEVICE)
    optimizer = torch.optim.AdamW([
        {"params": model.image_features.parameters(), "lr": settings["backbone_lr"]},
        {"params": [p for name, p in model.named_parameters() if not name.startswith("image_features.")], "lr": settings["head_lr"]},
    ], weight_decay=1e-4)
    y = labels[train_index]
    pos_weight = torch.tensor([max(float((y == 0).sum()) / max(int((y == 1).sum()), 1), .5)], device=DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_auc, best_epoch, stale = -1.0, 1, 0
    for epoch in range(1, epochs + 1):
        model.set_stage(epoch > settings["warmup_epochs"])
        model.train()
        shuffled = np.asarray(train_index)[torch.randperm(len(train_index)).numpy()]
        total_loss = 0.0
        for start in range(0, len(shuffled), settings["batch_size"]):
            index = shuffled[start:start + settings["batch_size"]]
            x, m = augment_batch(torch.from_numpy(images[index]), torch.from_numpy(masks[index]))
            yb = torch.as_tensor(labels[index], dtype=torch.float32, device=DEVICE)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = criterion(model(x.to(DEVICE), m.to(DEVICE)), yb)
            if not torch.isfinite(loss):
                raise ValueError("损失出现无效数值，请检查输入或降低学习率。")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(index)
        message = f"epoch {epoch:02d}/{epochs}, loss={total_loss / len(train_index):.4f}"
        if valid_index is not None:
            probability = predict(model, images[valid_index], masks[valid_index], settings["batch_size"])
            auc = float(roc_auc_score(labels[valid_index], probability))
            if auc > best_auc:
                best_auc, best_epoch, stale = auc, epoch, 0
            else:
                stale += 1
            message += f", inner_auc={auc:.4f}, best_epoch={best_epoch}"
        print(message, flush=True)
        if valid_index is not None and stale >= settings["patience"] and epoch > settings["warmup_epochs"]:
            break
    return model, best_epoch


def run_experiment(cases, images, masks, folder, settings, model_args, sequence="all", restart=False):
    folder.mkdir(parents=True, exist_ok=True)
    dev_index = np.flatnonzero(cases.split.eq("development"))
    dev = cases.iloc[dev_index].reset_index(drop=True)
    x, m = sequence_input(images[dev_index], sequence), masks[dev_index]
    y, folds = dev.label.to_numpy(dtype=int), dev.development_fold.to_numpy(dtype=int)
    if set(folds) != {1, 2, 3, 4, 5}:
        raise ValueError("开发队列需有1～5的固定折号。")
    oof, chosen_epochs, fold_rows, trained = np.zeros(len(dev), np.float32), [], [], False
    for fold in range(1, 6):
        train_index, valid_index = np.flatnonzero(folds != fold), np.flatnonzero(folds == fold)
        weight_file, prediction_file = folder / f"fold_{fold}.pt", folder / f"fold_{fold}_oof.csv"
        if not restart and weight_file.exists() and prediction_file.exists():
            saved = torch.load(weight_file, map_location="cpu", weights_only=True)
            previous = pd.read_csv(prediction_file, dtype={"mr": str}).set_index("mr")
            if not previous.index.is_unique or set(previous.index) != set(dev.mr.iloc[valid_index]):
                raise ValueError("已有折结果的患者不同；重训请设置RESTART=True。")
            previous = previous.loc[dev.mr.iloc[valid_index]]
            if not np.array_equal(previous.label, y[valid_index]):
                raise ValueError("已有折结果的标签不同；重训请设置RESTART=True。")
            oof[valid_index], epochs = previous.probability.to_numpy(), int(saved["epochs"])
            print(f"fold {fold}：读取已完成结果", flush=True)
        else:
            inner_train, inner_valid = train_test_split(train_index, test_size=.2, stratify=y[train_index], random_state=SEED + fold)
            probe, epochs = fit_network(x, m, y, inner_train, model_args, settings, settings["max_epochs"], SEED + fold, inner_valid)
            del probe
            model, _ = fit_network(x, m, y, train_index, model_args, settings, epochs, SEED + 100 + fold)
            oof[valid_index] = predict(model, x[valid_index], m[valid_index], settings["batch_size"])
            torch.save({"state_dict": {k: v.cpu() for k, v in model.state_dict().items()}, "model_args": model_args,
                        "sequence": sequence, "epochs": epochs, "settings": settings,
                        "train_mrs": dev.mr.iloc[train_index].tolist()}, weight_file)
            frame = dev.iloc[valid_index][["mr", "label", "development_fold"]].copy()
            frame["probability"] = oof[valid_index]
            frame.to_csv(prediction_file, index=False)
            del model
            trained = True
        chosen_epochs.append(epochs)
        fold_rows.append({"fold": fold, "auc": float(roc_auc_score(y[valid_index], oof[valid_index])), "epochs": epochs})
    threshold = choose_threshold(y, oof)
    final_file = folder / "model.pt"
    if restart or trained or not final_file.exists():
        final_epochs = max(1, int(np.median(chosen_epochs)))
        final, _ = fit_network(x, m, y, np.arange(len(dev)), model_args, settings, final_epochs, SEED + 1000)
        torch.save({"state_dict": {k: v.cpu() for k, v in final.state_dict().items()}, "model_args": model_args,
                    "sequence": sequence, "threshold": threshold, "epochs": final_epochs, "settings": settings,
                    "train_mrs": dev.mr.tolist()}, final_file)
        del final
    frame = dev[["mr", "label", "development_fold"]].copy()
    frame["probability"] = oof
    frame.to_csv(folder / "development_oof.csv", index=False)
    pd.DataFrame(fold_rows).to_csv(folder / "fold_metrics.csv", index=False)
    metrics = {**classification_metrics(y, oof, threshold), "mean_fold_auc": float(np.mean([r["auc"] for r in fold_rows]))}
    pd.DataFrame([metrics]).to_csv(folder / "development_metrics.csv", index=False)
    return pd.DataFrame([metrics])


In [ ]:
if RUN_TRAINING:
    display(run_experiment(cases, images, masks, OUTPUT, TRAIN, MODEL, restart=RESTART))
else:
    print("需要训练时将RUN_TRAINING改为True。")


## 临床对照与融合
临床对照使用年龄、受教育年限、NIHSS、同型半胱氨酸、尿酸；融合模型在这五项前加上影像模型的预测概率。MoCA 只定义结局，不参与输入。

融合沿用一轮折外预测再做逻辑回归的方式。它不是完整嵌套的融合评估，因此融合的开发集指标用于探索，最终效果以独立时间测试为准。02 重新训练后，也要重跑这一节，使融合模型对应当前影像模型。


In [ ]:
import joblib
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold


def clinical_search():
    pipeline = Pipeline([("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
                         ("scale", StandardScaler()), ("lr", LogisticRegression(solver="liblinear", max_iter=3000))])
    return GridSearchCV(pipeline, {"lr__C": [.003, .01, .03, .1, .3, 1.]}, scoring="roc_auc",
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED), n_jobs=1, error_score="raise")


def fit_clinical_models(cases, resnet_folder):
    dev = cases.query("split == 'development'").reset_index(drop=True)
    image_oof = pd.read_csv(resnet_folder / "development_oof.csv", dtype={"mr": str}).set_index("mr")
    if not image_oof.index.is_unique or set(image_oof.index) != set(dev.mr):
        raise ValueError("影像OOF与开发患者不一致。")
    if not np.array_equal(image_oof.loc[dev.mr, "label"], dev.label):
        raise ValueError("影像OOF的结局与患者清单不一致。")
    x_clinical = dev[CLINICAL].to_numpy(dtype=float)
    x_fused = np.column_stack([image_oof.loc[dev.mr, "probability"].to_numpy(), x_clinical])
    y, folds = dev.label.to_numpy(dtype=int), dev.development_fold.to_numpy(dtype=int)
    models, rows, tables = {}, [], []
    for name, values in (("Clinical5", x_clinical), ("ResNet_Clinical5", x_fused)):
        probabilities = np.zeros(len(dev), dtype=float)
        for fold in range(1, 6):
            valid = folds == fold
            search = clinical_search().fit(values[~valid], y[~valid])
            probabilities[valid] = search.predict_proba(values[valid])[:, 1]
        threshold = choose_threshold(y, probabilities)
        models[name] = {"model": clinical_search().fit(values, y).best_estimator_, "threshold": threshold}
        rows.append({"model": name, **classification_metrics(y, probabilities, threshold)})
        table = dev[["mr", "label", "development_fold"]].copy()
        table["model"], table["probability"] = name, probabilities
        tables.append(table)
    joblib.dump(models, resnet_folder / "clinical_models.joblib")
    pd.DataFrame(rows).to_csv(resnet_folder / "clinical_development_metrics.csv", index=False)
    pd.concat(tables).to_csv(resnet_folder / "clinical_development_oof.csv", index=False)
    return pd.DataFrame(rows)


In [ ]:
if RUN_CLINICAL:
    display(fit_clinical_models(cases, OUTPUT))
else:
    print("影像五折训练完成后，将RUN_CLINICAL改为True即可训练临床对照和融合。")


## 主要输出
`model.pt` 是最终影像模型，`clinical_models.joblib` 是临床及融合模型。`development_oof.csv` 保存患者级折外预测，`development_metrics.csv` 保存开发队列指标，`fold_1.pt`～`fold_5.pt` 用于 04 的折外解释。
